# CERT r4.2 Insider Threat — Stage 1: Streaming Preprocessing

Builds the **user-day behavioral evidence table** that feeds the Bayesian
Network, straight out of the source zip — no manual extraction, no writing
the ~84GB uncompressed dataset to disk.

**How it stays cheap on Colab Free:**
- The zip stays as one 8GB file on local disk.
- Each log CSV is opened *as a stream* from inside the zip
  (`zipfile.ZipFile(...).open(member)`) and read in chunks with
  `pandas.read_csv(..., chunksize=...)` — pandas decompresses and reads
  incrementally, nothing large ever sits fully in RAM or on disk.
- Only the columns each log type needs are kept (`usecols=...`) — free-text
  fields like `http.csv`'s page content or `email.csv`'s body are never
  loaded, which is what keeps an 84GB dataset tractable.
- Output is one small table: ~1000 users × ~500 days × a handful of
  numeric columns — tens of MB, easily fits in memory for every later stage.

Run cells top to bottom. Nothing here needs a GPU.

## 1. Mount Drive and locate the zip

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Point this at wherever you put the zip in Drive, e.g.:
DRIVE_ZIP_PATH = '/content/drive/MyDrive/cert_r4.2/r4.2.zip'
LOCAL_ZIP_PATH = '/content/r4.2.zip'


In [ ]:
import shutil, os

# Copy the 8GB zip onto local Colab disk once per session — much faster
# than reading zip members directly off a Drive mount, and Colab's local
# disk has plenty of room for an 8GB file.
if not os.path.exists(LOCAL_ZIP_PATH):
    shutil.copy(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)

print(f"local zip size: {os.path.getsize(LOCAL_ZIP_PATH) / 1e9:.2f} GB")


## 2. Inspect what's actually in the archive

CERT releases vary slightly in exact filenames/columns between distributions.
Run this first against your real zip and compare the printed column names
to the ones assumed in Section 3 below — adjust `usecols=[...]` there if
anything differs.

In [ ]:
import zipfile

with zipfile.ZipFile(LOCAL_ZIP_PATH) as zf:
    for info in zf.infolist():
        print(f"{info.filename:30s}  compressed={info.compress_size/1e6:8.1f}MB  "
              f"uncompressed={info.file_size/1e6:8.1f}MB")


In [ ]:
# Peek at the header + first few rows of each log file without extracting it
import pandas as pd

with zipfile.ZipFile(LOCAL_ZIP_PATH) as zf:
    for name in ["logon.csv", "device.csv", "file.csv", "email.csv", "http.csv"]:
        if name in zf.namelist():
            with zf.open(name) as f:
                preview = pd.read_csv(f, nrows=3)
            print(f"--- {name} ---")
            print(list(preview.columns))
            display(preview)


## 3. Streaming aggregator

Adjust `usecols` per function if Section 2's column inspection showed
different names in your release. `internal_domain` in `stream_email`
should match the org email domain used in your CERT release (commonly
`dtaa.com` for r4.2).

In [ ]:
"""
Streams CERT r4.2 log files directly out of the source zip (no extraction to
disk, no full-file load into memory) and builds a user-day behavioral
evidence table suitable for the Bayesian Network stage.

Only the columns each log type actually needs are kept — large free-text
fields (http 'content', email 'content'/body) are dropped at read time via
`usecols`, which is what keeps this cheap even though the archive expands
to ~80GB+ uncompressed.
"""
import zipfile
import pandas as pd
from collections import defaultdict

CHUNK_SIZE = 200_000  # rows per chunk; tune down if a Colab session is RAM-tight

# after-hours window: local convention used across CERT literature (9am-5pm business hours)
BUSINESS_START_HOUR = 9
BUSINESS_END_HOUR = 17


def _is_after_hours(ts_series):
    hours = ts_series.dt.hour
    return (hours < BUSINESS_START_HOUR) | (hours >= BUSINESS_END_HOUR)


def stream_logon(zf, agg):
    with zf.open("logon.csv") as f:
        for chunk in pd.read_csv(f, usecols=["date", "user", "activity"], chunksize=CHUNK_SIZE):
            chunk["date"] = pd.to_datetime(chunk["date"])
            chunk["day"] = chunk["date"].dt.date
            logons = chunk[chunk["activity"] == "Logon"].copy()
            logons["after_hours"] = _is_after_hours(logons["date"])
            grp = logons.groupby(["user", "day"])
            for (user, day), g in grp:
                key = (user, day)
                agg[key]["logon_count"] += len(g)
                agg[key]["after_hours_logon_count"] += int(g["after_hours"].sum())


def stream_device(zf, agg):
    with zf.open("device.csv") as f:
        for chunk in pd.read_csv(f, usecols=["date", "user", "activity"], chunksize=CHUNK_SIZE):
            chunk["date"] = pd.to_datetime(chunk["date"])
            chunk["day"] = chunk["date"].dt.date
            connects = chunk[chunk["activity"] == "Connect"]
            grp = connects.groupby(["user", "day"]).size()
            for (user, day), cnt in grp.items():
                agg[(user, day)]["device_connect_count"] += int(cnt)


def stream_file(zf, agg):
    with zf.open("file.csv") as f:
        cols = ["date", "user", "activity", "to_removable_media"]
        for chunk in pd.read_csv(f, usecols=cols, chunksize=CHUNK_SIZE):
            chunk["date"] = pd.to_datetime(chunk["date"])
            chunk["day"] = chunk["date"].dt.date
            grp = chunk.groupby(["user", "day"])
            for (user, day), g in grp:
                key = (user, day)
                agg[key]["file_activity_count"] += len(g)
                to_removable = g["to_removable_media"].astype(str).str.lower().isin(["true", "1"])
                agg[key]["file_copy_to_removable_count"] += int(to_removable.sum())


def stream_email(zf, agg, internal_domain="dtaa.com"):
    with zf.open("email.csv") as f:
        cols = ["date", "user", "to", "size"]
        for chunk in pd.read_csv(f, usecols=cols, chunksize=CHUNK_SIZE):
            chunk["date"] = pd.to_datetime(chunk["date"])
            chunk["day"] = chunk["date"].dt.date
            chunk["to"] = chunk["to"].fillna("")
            chunk["is_external"] = ~chunk["to"].str.contains(internal_domain, na=False)
            grp = chunk.groupby(["user", "day"])
            for (user, day), g in grp:
                key = (user, day)
                agg[key]["email_count"] += len(g)
                agg[key]["external_email_count"] += int(g["is_external"].sum())
                agg[key]["email_size_total"] += float(g["size"].sum())


def stream_http(zf, agg):
    with zf.open("http.csv") as f:
        cols = ["date", "user"]
        for chunk in pd.read_csv(f, usecols=cols, chunksize=CHUNK_SIZE):
            chunk["date"] = pd.to_datetime(chunk["date"])
            chunk["day"] = chunk["date"].dt.date
            grp = chunk.groupby(["user", "day"]).size()
            for (user, day), cnt in grp.items():
                agg[(user, day)]["web_activity_count"] += int(cnt)


FEATURE_COLUMNS = [
    "logon_count", "after_hours_logon_count",
    "device_connect_count",
    "file_activity_count", "file_copy_to_removable_count",
    "email_count", "external_email_count", "email_size_total",
    "web_activity_count",
]


def build_user_day_table(zip_path):
    agg = defaultdict(lambda: defaultdict(float))
    with zipfile.ZipFile(zip_path) as zf:
        names = set(zf.namelist())
        if "logon.csv" in names:
            stream_logon(zf, agg)
        if "device.csv" in names:
            stream_device(zf, agg)
        if "file.csv" in names:
            stream_file(zf, agg)
        if "email.csv" in names:
            stream_email(zf, agg)
        if "http.csv" in names:
            stream_http(zf, agg)

    rows = []
    for (user, day), feats in agg.items():
        row = {"user": user, "day": day}
        row.update({c: feats.get(c, 0) for c in FEATURE_COLUMNS})
        rows.append(row)

    df = pd.DataFrame(rows, columns=["user", "day"] + FEATURE_COLUMNS)
    df = df.sort_values(["user", "day"]).reset_index(drop=True)
    return df


## 4. Run it against the real zip

In [ ]:
df = build_user_day_table(LOCAL_ZIP_PATH)
print(f"{len(df)} user-day rows, {df['user'].nunique()} users, "
      f"{df['day'].min()} to {df['day'].max()}")
df.head(10)


## 5. Save the small output back to Drive

This is the file every later stage (thresholding, discretization, BN
structure/CPTs, inference) works from — the raw logs aren't touched again.

In [ ]:
OUT_PATH = '/content/drive/MyDrive/cert_r4.2/user_day_features.csv'
df.to_csv(OUT_PATH, index=False)
print(f"saved {OUT_PATH}  ({os.path.getsize(OUT_PATH)/1e6:.1f} MB)")


## 6. Optional: quick sanity plots

Skip if you'd rather move straight to feature thresholding / discretization
in the next notebook.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df['logon_count'], bins=30); axes[0].set_title('logon_count')
axes[1].hist(df['after_hours_logon_count'], bins=30); axes[1].set_title('after_hours_logon_count')
axes[2].hist(df['external_email_count'], bins=30); axes[2].set_title('external_email_count')
plt.tight_layout()
plt.show()
